# 11 — Análise de Resultados: RAG (BM25, Vector e Hybrid)

Avalia as variantes RAG no split de validação (20% do dataset).

**Importante:** O conjunto de avaliação (~4.200 tweets) é diferente dos outros experimentos (~20.813). As métricas não são diretamente comparáveis — o split foi feito para garantir separação limpa entre corpus de retrieval e avaliação.

Variantes:
- `rag_bm25_k3` — BM25 sparse retrieval, top-3 exemplos
- `rag_vector_k3` — Dense retrieval (paraphrase-multilingual-MiniLM-L12-v2), top-3 exemplos
- `rag_hybrid_qdrant_k3` — Hybrid search Qdrant: dense + TF-IDF sparse, fusão via RRF, top-3 exemplos

In [ ]:
from pathlib import Path
import polars as pl
from sklearn.metrics import classification_report, f1_score

ROOT = Path("..")
RESULTS = ROOT / "results" / "full"

VARIANTS = {
    "rag_bm25_k3": "RAG-BM25 (K=3)",
    "rag_vector_k3": "RAG-Vector MiniLM (K=3)",
    "rag_hybrid_qdrant_k3": "RAG-Hybrid Qdrant RRF (K=3)",
}

In [ ]:
summary = []

for key, name in VARIANTS.items():
    path = RESULTS / f"{key}.csv"
    if not path.exists():
        print(f"[SKIP] {path} não encontrado")
        continue

    df = pl.read_csv(path)
    y_true = df["label"].to_list()
    y_pred = df["predicao"].to_list()

    f1_macro   = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    unknown     = (df["predicao"] == "unknown").sum()

    summary.append({"variante": name, "f1_macro": round(f1_macro, 4), "f1_weighted": round(f1_weighted, 4), "unknown": unknown, "tweets": len(df)})

    print(f"\n{'='*60}")
    print(f"  {name}  |  F1-macro: {f1_macro:.4f}  |  F1-weighted: {f1_weighted:.4f}  |  unknown: {unknown}")
    print(f"{'='*60}")
    print(classification_report(y_true, y_pred, zero_division=0))

In [ ]:
print("\n=== Resumo Comparativo (val set 20%) ===")
summary_df = pl.DataFrame(summary).sort("f1_macro", descending=True)
print(summary_df)

## Nota sobre comparabilidade

Os resultados do dataset completo (experimentos 04-08) foram calculados em 20.813 tweets sem split.
Aqui avaliamos em ~4.163 tweets (20% estratificado).

Para comparação orientativa com o melhor resultado anterior:
- **FS-v2 2ex+Antibias** no full: F1-macro = **0.3173**

Classes raras no val (esperado):
- `racism`: ~4 exemplos
- `xenophobia`: ~6 exemplos  
- `misogyny`: ~9 exemplos

O F1-macro dessas classes será ruidoso devido ao baixo suporte.